# Détection d'intrusions réseau par apprentissage automatique
**Notebook de départ : NSL-KDD** — Licence 3 TCO, ASJA Antsirabe

Plan : 1) chargement, 2) exploration, 3) prétraitement, 4) modèles, 5) comparaison, 6) analyse des erreurs.
Chaque cellule est commentée : l'objectif est que tu puisses **expliquer chaque ligne** devant le jury.

## 0. Imports et configuration

In [ ]:
import time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, ConfusionMatrixDisplay, classification_report)
warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED)

# Source des fichiers : copie GitHub du dataset NSL-KDD.
# Hors ligne ? Télécharge KDDTrain+.txt et KDDTest+.txt puis mets leur chemin local ici.
URL_TRAIN = "https://raw.githubusercontent.com/defcom17/NSL_KDD/master/KDDTrain+.txt"
URL_TEST  = "https://raw.githubusercontent.com/defcom17/NSL_KDD/master/KDDTest+.txt"

## 1. Chargement des données
NSL-KDD : 41 caractéristiques par connexion, une étiquette (`normal` ou un type d'attaque) et un niveau de difficulté.
Le fichier n'a pas d'en-tête : on nomme les colonnes nous-mêmes.

In [ ]:
cols = ["duration","protocol_type","service","flag","src_bytes","dst_bytes","land","wrong_fragment","urgent",
        "hot","num_failed_logins","logged_in","num_compromised","root_shell","su_attempted","num_root",
        "num_file_creations","num_shells","num_access_files","num_outbound_cmds","is_host_login","is_guest_login",
        "count","srv_count","serror_rate","srv_serror_rate","rerror_rate","srv_rerror_rate","same_srv_rate",
        "diff_srv_rate","srv_diff_host_rate","dst_host_count","dst_host_srv_count","dst_host_same_srv_rate",
        "dst_host_diff_srv_rate","dst_host_same_src_port_rate","dst_host_srv_diff_host_rate","dst_host_serror_rate",
        "dst_host_srv_serror_rate","dst_host_rerror_rate","dst_host_srv_rerror_rate","label","difficulty"]
train = pd.read_csv(URL_TRAIN, names=cols)
test  = pd.read_csv(URL_TEST,  names=cols)
print("Train :", train.shape, "| Test :", test.shape)
train.head()

## 2. Exploration
Questions à te poser : quelles classes ? équilibrées ? valeurs manquantes ? doublons ?

In [ ]:
print("Valeurs manquantes (train) :", int(train.isna().sum().sum()))
print("Doublons (train) :", int(train.duplicated().sum()))
print("Types d'attaque (train) :", train.label.nunique(), "| (test) :", test.label.nunique())
print("Attaques présentes dans le test mais absentes du train :",
      sorted(set(test.label) - set(train.label)))

In [ ]:
# Binaire : normal (0) vs attaque (1)
for df in (train, test):
    df["is_attack"] = (df.label != "normal").astype(int)

fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
for a, (name, df) in zip(ax, [("Train", train), ("Test", test)]):
    df.is_attack.value_counts().sort_index().plot.bar(ax=a, color=["tab:green", "tab:red"])
    a.set_title(f"{name} : 0 = normal, 1 = attaque"); a.set_xlabel("")
plt.tight_layout(); plt.show()
print(train.is_attack.value_counts(normalize=True).round(3))

### Regroupement en 4 familles d'attaques (classification de la famille)
Les 4 familles classiques : **DoS**, **Probe** (reconnaissance/scan), **R2L** (accès distant non autorisé), **U2R** (élévation de privilèges).
Utile pour la partie « type d'attaque probable » de ta démo.

In [ ]:
families = {
 "DoS":   ["back","land","neptune","pod","smurf","teardrop","apache2","mailbomb","processtable","udpstorm","worm"],
 "Probe": ["ipsweep","nmap","portsweep","satan","mscan","saint"],
 "R2L":   ["ftp_write","guess_passwd","imap","multihop","phf","spy","warezclient","warezmaster",
           "sendmail","named","snmpgetattack","snmpguess","xlock","xsnoop","httptunnel"],
 "U2R":   ["buffer_overflow","loadmodule","perl","rootkit","ps","sqlattack","xterm"],
}
lab2fam = {l: f for f, ls in families.items() for l in ls}
lab2fam["normal"] = "normal"
for df in (train, test):
    df["family"] = df.label.map(lab2fam).fillna("autre")
print(train.family.value_counts())
print(test.family.value_counts())

## 3. Prétraitement
- **Encodage one-hot** des 3 variables catégorielles (`protocol_type`, `service`, `flag`)
- **Normalisation** (StandardScaler) des variables numériques
- On met tout dans un `Pipeline` : le scaler est ajusté **uniquement sur le train**, ce qui évite la fuite de données (data leakage).

In [ ]:
cat_cols = ["protocol_type", "service", "flag"]
drop_cols = ["label", "difficulty", "is_attack", "family"]
num_cols = [c for c in train.columns if c not in cat_cols + drop_cols]

X_train, X_test = train.drop(columns=drop_cols), test.drop(columns=drop_cols)
y_train, y_test = train.is_attack, test.is_attack

prep = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
])
print(f"{len(num_cols)} variables numériques + {len(cat_cols)} catégorielles")

## 4. Modèles de base
On compare 3 modèles supervisés et 1 modèle non supervisé (Isolation Forest, entraîné **uniquement sur le trafic normal**).
Pour chaque modèle on mesure les métriques et les temps d'entraînement/prédiction.

In [ ]:
def evaluate(name, y_true, y_pred, t_fit, t_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return dict(modele=name,
                accuracy=accuracy_score(y_true, y_pred),
                precision=precision_score(y_true, y_pred),
                rappel=recall_score(y_true, y_pred),
                f1=f1_score(y_true, y_pred),
                faux_positifs=fp, faux_negatifs=fn,
                t_entrainement_s=t_fit, t_prediction_s=t_pred)

models = {
    "Régression logistique": LogisticRegression(max_iter=1000, random_state=SEED),
    "Arbre de décision":     DecisionTreeClassifier(max_depth=15, random_state=SEED),
    "Random Forest":         RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=SEED),
}
results, preds = [], {}
for name, clf in models.items():
    pipe = Pipeline([("prep", prep), ("clf", clf)])
    t0 = time.time(); pipe.fit(X_train, y_train); t_fit = time.time() - t0
    t0 = time.time(); y_pred = pipe.predict(X_test); t_pred = time.time() - t0
    preds[name] = y_pred
    results.append(evaluate(name, y_test, y_pred, t_fit, t_pred))
    print("OK :", name)

In [ ]:
# Isolation Forest : apprend ce qu'est le trafic « normal », tout le reste est une anomalie
Xtr_t = prep.fit_transform(X_train)
Xte_t = prep.transform(X_test)
normal_only = Xtr_t[(y_train == 0).values]

iso = IsolationForest(n_estimators=200, contamination=0.05, random_state=SEED, n_jobs=-1)
t0 = time.time(); iso.fit(normal_only); t_fit = time.time() - t0
t0 = time.time(); y_pred = (iso.predict(Xte_t) == -1).astype(int); t_pred = time.time() - t0
preds["Isolation Forest"] = y_pred
results.append(evaluate("Isolation Forest", y_test, y_pred, t_fit, t_pred))

## 5. Comparaison des résultats
Regarde surtout **rappel** (attaques détectées), **précision** (alertes justifiées) et **faux négatifs** (attaques ratées, les plus graves).

In [ ]:
res = pd.DataFrame(results).set_index("modele").round(4)
res

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
res[["accuracy", "precision", "rappel", "f1"]].plot.bar(ax=ax)
ax.set_ylim(0, 1); ax.set_xlabel(""); ax.set_title("Comparaison sur le jeu de TEST (NSL-KDD)")
plt.xticks(rotation=15); plt.legend(loc="lower right"); plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, len(preds), figsize=(4.2 * len(preds), 3.8))
for a, (name, y_pred) in zip(axes, preds.items()):
    ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred), display_labels=["normal", "attaque"]).plot(ax=a, colorbar=False)
    a.set_title(name, fontsize=10)
plt.tight_layout(); plt.show()

## 6. Analyse des erreurs : quelles attaques sont ratées ?
Un bon rapport ne s'arrête pas aux scores globaux : on regarde **quelles familles** échappent au modèle.

In [ ]:
best = "Random Forest"
tmp = test.assign(pred=preds[best])
att = tmp[tmp.is_attack == 1]
rappel_famille = att.groupby("family").apply(lambda d: (d.pred == 1).mean()).round(3)
print(f"Rappel par famille d'attaque ({best}) :")
print(rappel_famille)
print()
print("Attaques du test ratées le plus souvent :")
manq = att[att.pred == 0].label.value_counts().head(10)
print(manq)

In [ ]:
# Variables les plus importantes (Random Forest)
rf_pipe = Pipeline([("prep", prep), ("clf", RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=SEED))]).fit(X_train, y_train)
names = rf_pipe.named_steps["prep"].get_feature_names_out()
imp = pd.Series(rf_pipe.named_steps["clf"].feature_importances_, index=names).sort_values(ascending=False).head(15)
imp.iloc[::-1].plot.barh(figsize=(7, 4.5), title="Top 15 variables (Random Forest)")
plt.tight_layout(); plt.show()

## 7. Pistes pour la suite (à faire pour le mémoire)
1. **Validation croisée** sur le train (`StratifiedKFold`) en plus du test officiel
2. **Réglage d'hyperparamètres** (`GridSearchCV` / `RandomizedSearchCV`), évalué sur validation, pas sur le test
3. **Classification multiclasse** (4 familles) avec `classification_report`, et rappel par famille pour U2R/R2L (très rares)
4. **Déséquilibre** : `class_weight="balanced"`, SMOTE
5. **Généralisation** : entraîner sur NSL-KDD, tester sur CICIDS2017 (attention : colonnes différentes, il faut une table de correspondance)
6. **Comparaison avec tes règles** Wazuh/Sigma (baseline à base de règles) : c'est ta problématique
7. **Sauvegarde du modèle** (`joblib.dump`) puis démo Streamlit

> Note honnête pour le jury : le jeu de test NSL-KDD contient des attaques absentes du train, donc un score plus bas qu'en validation croisée est **normal** et mérite d'être expliqué.